In [1]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

2026-07-28 12:12:03.867972: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/lucifer666/anaconda3/envs/py312/lib/python3.12/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


In [5]:
builder = tfds.builder("mnist")

builder.download_and_prepare()

ds_train = builder.as_dataset(split="train", as_supervised=True)
ds_test = builder.as_dataset(split="test", as_supervised=True)

Dl Completed...: 0 url [00:00, ? url/s]
Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]2026-07-28 12:26:22.664097: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Dataset mnist downloaded and prepared to /home/lucifer666/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.


In [7]:
(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True
)

In [8]:
def normalize_image(image, label):
    return tf.cast(image, tf.float32) / 255. , label

In [9]:
ds_train = ds_train.map(normalize_image, num_parallel_calls=tf.data.AUTOTUNE)
ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(128)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

In [10]:
ds_test = ds_test.map(normalize_image, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.batch(128)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)

In [11]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

/home/lucifer666/anaconda3/envs/py312/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [13]:
model.fit(ds_train, epochs=20, validation_data=ds_test)

Epoch 1/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0522 - sparse_categorical_accuracy: 0.9851 - val_loss: 0.0832 - val_sparse_categorical_accuracy: 0.9750
Epoch 2/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0448 - sparse_categorical_accuracy: 0.9871 - val_loss: 0.0773 - val_sparse_categorical_accuracy: 0.9775
Epoch 3/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0378 - sparse_categorical_accuracy: 0.9895 - val_loss: 0.0779 - val_sparse_categorical_accuracy: 0.9761
Epoch 4/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0322 - sparse_categorical_accuracy: 0.9913 - val_loss: 0.0766 - val_sparse_categorical_accuracy: 0.9777
Epoch 5/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0278 - sparse_categorical_accuracy: 0.9923 - val_loss: 0.0722 - val_sparse_categorical_accuracy: 0.9793
Epoch 6/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0228 - sparse_categorical_accuracy: 0.9942 - val_loss: 0.0742 - val_sparse_categorical_accuracy: 0.9781
Epoc

In [14]:
from sklearn.metrics import classification_report

In [15]:
y_pred = model.predict(ds_test)
y_hat = tf.argmax(y_pred, axis=1)

79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [16]:
print(classification_report(y_true=np.concatenate([y for x, y in ds_test], axis=0), y_pred=y_hat.numpy()))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99       980
           1       0.99      0.99      0.99      1135
           2       0.97      0.98      0.97      1032
           3       0.95      0.98      0.97      1010
           4       0.99      0.96      0.98       982
           5       0.97      0.97      0.97       892
           6       0.98      0.98      0.98       958
           7       0.97      0.98      0.97      1028
           8       0.97      0.96      0.96       974
           9       0.97      0.96      0.97      1009

    accuracy                           0.98     10000
   macro avg       0.98      0.97      0.97     10000
weighted avg       0.98      0.98      0.98     10000



2026-07-28 12:30:48.929992: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
model.save('./model/mnist_model.h5')